<a href="https://colab.research.google.com/github/manan36chauhan/data_science_project/blob/First-init/sdg_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('mnist.csv')
df

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,25268,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3996,6473,6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3997,5821,7,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3998,1751,9,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df.shape

(4000, 786)

In [5]:
from sklearn.model_selection import train_test_split

# Define X (features) and y (target)
X = df.drop(['id', 'class'], axis=1)
y = df['class']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

# Create a pipeline with StandardScaler and SGDClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('sgdclassifier', SGDClassifier(random_state=42))
])

# Define the parameter grid for GridSearchCV
param_grid = {
    'sgdclassifier__loss': ['hinge', 'log_loss', 'modified_huber'],
    'sgdclassifier__penalty': ['l2', 'l1', 'elasticnet'],
    'sgdclassifier__alpha': [0.0001, 0.001, 0.01]
}

# Perform GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Print the best parameters and best score
print("Best parameters: ", grid_search.best_params_)
print("Best cross-validation accuracy: ", grid_search.best_score_)

# Get the best model from the grid search
best_model = grid_search.best_estimator_

Best parameters:  {'sgdclassifier__alpha': 0.0001, 'sgdclassifier__loss': 'hinge', 'sgdclassifier__penalty': 'l2'}
Best cross-validation accuracy:  0.8694029850746269


In [8]:
# sgd_mnist_pipeline.py
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from scipy.stats import uniform
import joblib
import json
import time

# ---------- Config ----------
CSV_PATH = "/content/mnist.csv"
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_JOBS = -1
N_ITER = 40
CV_FOLDS = 3
OUT_MODEL = "sgd_mnist_best.joblib"
# ----------------------------

# 1) Load data
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at {CSV_PATH}. Place your file there or change CSV_PATH.")

print("Loading CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)

# Drop 'id' column if present and extract label
if 'id' in df.columns:
    df = df.drop(columns=['id'])

if 'class' not in df.columns:
    raise ValueError("CSV must contain a 'class' column for labels.")

y = df['class'].values
X = df.drop(columns=['class']).values  # pixel1..pixel784

print("X shape:", X.shape, "y shape:", y.shape)
print("Unique labels:", np.unique(y))

# 2) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# 3) Pipeline: impute -> scale -> SGD
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("sgd", SGDClassifier(random_state=RANDOM_STATE, max_iter=1000, tol=1e-3))
])

# 4) Hyperparameter search - RandomizedSearchCV
# Use sensible distributions; tweak as needed
param_distributions = {
    "sgd__loss": ["log_loss", "hinge"],          # logistic (prob) or hinge (SVM-like)
    "sgd__penalty": ["l2", "elasticnet"],        # try l1 if you want sparse weights
    "sgd__alpha": uniform(loc=1e-6, scale=1e-2), # continuous in this range
    "sgd__learning_rate": ["optimal", "invscaling", "constant"],
    "sgd__eta0": uniform(loc=0.0, scale=0.1),   # initial learning rate (used with invscaling/constant)
    # note: l1_ratio is used if penalty='elasticnet' -> we will add it below conditionally if needed
}

# Custom CV - stratified
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="accuracy",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=2
)

print("Starting RandomizedSearchCV (this may take time for full MNIST)...")
t0 = time.time()
search.fit(X_train, y_train)
t1 = time.time()
print(f"RandomizedSearchCV done in {(t1 - t0)/60:.2f} minutes")

print("Best CV score:", search.best_score_)
print("Best params:", search.best_params_)

# If best uses elasticnet, you may want to tune l1_ratio separately:
if search.best_params_.get("sgd__penalty") == "elasticnet":
    print("Consider a follow-up search tuning 'sgd__l1_ratio' in [0.0, 1.0].")

best_pipeline = search.best_estimator_

# 5) Evaluate on test set
y_pred = best_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
print(f"\nTest accuracy: {acc:.4f}")
print(f"Test macro-F1: {f1_macro:.4f}\n")
print(classification_report(y_test, y_pred))

# Confusion matrix (you can save or plot it)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix shape:", cm.shape)

# Save best pipeline
joblib.dump(best_pipeline, OUT_MODEL)
print("Saved best pipeline to", OUT_MODEL)

# Save search results (tight summary)
summary = {
    "best_score": float(search.best_score_),
    "best_params": search.best_params_,
    "test_accuracy": float(acc),
    "test_macro_f1": float(f1_macro)
}
with open("sgd_mnist_search_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved summary to sgd_mnist_search_summary.json")

Loading CSV: /content/mnist.csv
X shape: (4000, 784) y shape: (4000,)
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Starting RandomizedSearchCV (this may take time for full MNIST)...
Fitting 3 folds for each of 40 candidates, totalling 120 fits
RandomizedSearchCV done in 4.43 minutes
Best CV score: 0.8806262466056279
Best params: {'sgd__alpha': np.float64(0.008022969807540395), 'sgd__eta0': np.float64(0.007455064367977083), 'sgd__learning_rate': 'constant', 'sgd__loss': 'log_loss', 'sgd__penalty': 'elasticnet'}
Consider a follow-up search tuning 'sgd__l1_ratio' in [0.0, 1.0].

Test accuracy: 0.8900
Test macro-F1: 0.8872

              precision    recall  f1-score   support

           0       0.95      0.97      0.96        75
           1       0.95      0.97      0.96        97
           2       0.89      0.94      0.91        78
           3       0.86      0.86      0.86        84
           4       0.85      0.93      0.89        74
           5       0.79      0.82      0.81        73
 